# 01 · EDA de vivienda en Euskadi

**Unidad:** municipio y trimestre de inicio del contrato. **Objetivo observado:** renta mensual media EMAL. No son anuncios individuales ni expedientes de ayudas.

Este proyecto orienta a servicios municipales de juventud. La predicción es supervisada y de regresión; no es una resolución administrativa. Los originales se capturan con `scripts/capture_sources.py` y se verifican mediante SHA-256.

In [ ]:
from pathlib import Path
import sys, os, json
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').exists())
sys.path.insert(0, str(ROOT))
OUT = Path(os.getenv('RETO1_ARTIFACT_DIR', str(ROOT / 'entrega_ml')))
from src.ml_data import read_emal, temporal_split, FEATURES, TARGET
from src.ml_workflow import make_eda
frame, procedencia = read_emal(ROOT / 'data/capturas_oficiales')
train, val, test = temporal_split(frame)
if (OUT / 'data/procedencia.json').exists():
    anterior = json.loads((OUT / 'data/procedencia.json').read_text())
    assert anterior['fuente']['sha256'] == procedencia['fuente']['sha256'], 'La fuente ha cambiado: no mezclar entregas'
display(procedencia)
display(train.head())

## Campos y calidad

`codigo_municipio` conserva cinco dígitos; `municipio` y `provincia` identifican el territorio; `anio` y `trimestre` definen la observación; `numero_contratos` mide fianzas del trimestre; `renta_media_eur` es el objetivo. `renta_lag1`, `renta_lag4` y `contratos_lag1` son datos de trimestres anteriores, nunca del trimestre a predecir.

Se excluyen totales territoriales, stocks vigentes actualizados por IRAV y Galdakao por el cambio de perímetro. El guion de EMAL indica falta de publicación/oferta suficiente, no cero. Los objetivos ausentes se excluyen, **no se imputan**. No se eliminan rentas altas arbitrariamente.

In [ ]:
display(pd.DataFrame({'tipo': train.dtypes.astype(str), 'nulos_pct_train': train.isna().mean() * 100}))
assert not frame.duplicated(['codigo_municipio', 'periodo']).any()
print('Filas train / validación / test reservado:', len(train), len(val), len(test))
print('Variables de entrada:', FEATURES)
print('Objetivo:', TARGET)
print('Duplicados municipio/trimestre:', frame.duplicated(['codigo_municipio', 'periodo']).sum())

## Nulos y separación temporal

Los primeros retardos son nulos por construcción y la supresión de EMAL depende de la oferta: no suponemos ausencia al azar. `SimpleImputer` y `StandardScaler` se ajustan dentro de cada fold; `OneHotEncoder(handle_unknown='ignore')` mantiene la inferencia estable.

Entrenamiento: hasta 2023. Validación: 2024. Test reservado: desde 2025. La serie fue revisada en la fecha de captura: esta es una evaluación retrospectiva, no una reconstrucción de lo que se sabía en cada fecha histórica.

In [ ]:
notas = make_eda(train, OUT / 'resultados')
for archivo, conclusion in notas.items():
    if archivo.endswith('.png'):
        display(Image(filename=str(OUT / 'resultados' / archivo)))
    display(Markdown(conclusion))

## Conclusión del EDA

La persistencia temporal y las diferencias entre municipios justifican comparar un baseline de renta anterior con modelos supervisados. No utilizamos renta actual, precio/m² actual ni categorías creadas a partir del objetivo como entradas. El EDA comentado se basa en entrenamiento; el test no se emplea para decidir transformaciones.